In [5]:
import torch
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid
import torch.nn.functional as F
import torch.nn as nn

import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import SplineConv
from torch_geometric.typing import WITH_TORCH_SPLINE_CONV


import itertools
import os
#from utils import *


os.environ["DGLBACKEND"] = "pytorch"

import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import torch.nn.utils.prune as prune


In [2]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
sys.path.append(os.path.abspath("../"))

import torch


In [15]:

if not WITH_TORCH_SPLINE_CONV:
    quit("This example requires 'torch-spline-conv'")

dataset = 'Cora'
transform = T.Compose([
    T.RandomNodeSplit(num_val=500, num_test=500),
    T.TargetIndegree(),
])
path =  'data'
dataset = Planetoid(path, dataset, transform=transform)
data = dataset[0]

In [28]:


class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SplineConv(dataset.num_features, 16, dim=1, kernel_size=2)
        self.conv2 = SplineConv(16, dataset.num_classes, dim=1, kernel_size=2)

    def forward(self):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        x = F.dropout(x, training=self.training)
        x = F.elu(self.conv1(x, edge_index, edge_attr))
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index, edge_attr)
        return F.log_softmax(x, dim=1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, data = Net().to(device), data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-3)

In [17]:


def train(callbacks = None):
    model.train()
    optimizer.zero_grad()
    F.nll_loss(model()[data.train_mask], data.y[data.train_mask]).backward()
    optimizer.step()
    if callbacks is not None:
        for callback in callbacks:
                callback()


@torch.no_grad()
def test(model):
    model.eval()
    log_probs, accs = model(), []
    for _, mask in data('train_mask', 'test_mask'):
        pred = log_probs[mask].max(1)[1]
        acc = pred.eq(data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs



In [29]:
for epoch in range(5):
    train(callbacks = None)
    train_acc, test_acc = test(model)
    #if epoch % 10 == 0:
    print(f'Epoch: {epoch:03d}, Train: {train_acc:.4f}, Test: {test_acc:.4f}')

Epoch: 000, Train: 0.5872, Test: 0.5740
Epoch: 001, Train: 0.6153, Test: 0.5800
Epoch: 002, Train: 0.6633, Test: 0.6360
Epoch: 003, Train: 0.7313, Test: 0.6960
Epoch: 004, Train: 0.8074, Test: 0.7560


In [34]:
def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=True) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

def fine_grained_prune(tensor: torch.Tensor, sparsity : float) -> torch.Tensor:
    """
    magnitude-based pruning for single tensor
    :param tensor: torch.(cuda.)Tensor, weight of conv/fc layer
    :param sparsity: float, pruning sparsity
        sparsity = #zeros / #elements = 1 - #nonzeros / #elements
    :return:
        torch.(cuda.)Tensor, mask for zeros
    """
    sparsity = min(max(0.0, sparsity), 1.0)
    if sparsity == 1.0:
        tensor.zero_()
        return torch.zeros_like(tensor)
    elif sparsity == 0.0:
        return torch.ones_like(tensor)

    num_elements = tensor.numel()

    num_zeros = round(num_elements * sparsity)
    importance = tensor.abs()
    threshold = importance.view(-1).kthvalue(num_zeros).values
    mask = torch.gt(importance, threshold)
    tensor.mul_(mask)

    return mask

class FineGrainedPruner:
    def __init__(self, model, sparsity_dict):
        self.masks = FineGrainedPruner.prune(model, sparsity_dict)

    @torch.no_grad()
    def apply(self, model):
        for name, param in model.named_parameters():
            if name in self.masks:
                param *= self.masks[name]

    @staticmethod
    @torch.no_grad()
    def prune(model, sparsity_dict):
        masks = dict()
        for name, param in model.named_parameters():
            if param.dim() > 1: # we only prune conv and fc weights
                if isinstance(sparsity_dict, dict):
                    masks[name] = fine_grained_prune(param, sparsity_dict[name])
                else:
                    assert(sparsity_dict < 1 and sparsity_dict >= 0)
                    if sparsity_dict > 0:
                        masks[name] = fine_grained_prune(param, sparsity_dict)
        return masks

In [30]:
best_checkpoint = dict()
best_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
model.load_state_dict(best_checkpoint['state_dict'])
recover_model = lambda: model.load_state_dict(best_checkpoint['state_dict'])

In [31]:
t0=time.time()
train_acc, base_model_accuracy=test(model)
t1=time.time()
t_base_model=t1 - t0
###
base_model_size = get_model_size(model, count_nonzero_only=True)
num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"base model has accuracy on test set={base_model_accuracy:.2f}%")
print(f"base model has size={base_model_size/MiB:.2f} MiB")
print(f"The time inference of base model is ={t_base_model}") 
print(f"The number of parametrs of base model is:{num_parm_base_model}")

base model has accuracy on test set=0.76%
base model has size=0.26 MiB
The time inference of base model is =23.097032070159912
The number of parametrs of base model is:69143


# Let's Prune the Model and Re-Evaluate the Accuracy.

In [32]:
sparsity = 0.9
recover_model()
pruner = FineGrainedPruner(model, sparsity)
pruner.apply(model)



In [35]:

t0=time.time()
train_acc, pruned_model_accuracy=test(model)

t1=time.time()
t_pruned_model=t1 - t0
###
pruned_model_size = get_model_size(model)
num_parm_pruned_model=get_num_parameters(model, count_nonzero_only=False)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_model}")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")

90.0% sparse model has accuracy on test set=0.59%
90.0% sparse model has size=0.03 MiB
The time inference of 90.0% sparse model is =22.899877071380615
The number of parametrs of 90.0% sparse model is:69143
90.0% sparse model has size=0.03 MiB, which is 9.97X smaller than the 0.26 MiB dense model


## Finetuning Fine-grained Pruned Sparse Model

In [26]:


best_sparse_checkpoint = dict()
best_sparse_accuracy = 0
num_finetune_epochs=10
print(f'Finetuning Fine-grained Pruned Sparse Model')
for epoch in range(num_finetune_epochs):
    # At the end of each train iteration, we have to apply the pruning mask
    #    to keep the model sparse during the training

    train(callbacks=[lambda: pruner.apply(model)])
    train_acc, accuracy = test(model)
    
  
    is_best = accuracy > best_sparse_accuracy
    if is_best:
        best_sparse_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
        best_sparse_accuracy = accuracy
    
    if epoch % 20 == 0:
         print(f'    Epoch {epoch} Sparse Accuracy {accuracy:.2f}% / Best Sparse Accuracy: {best_sparse_accuracy:.2f}%')


Finetuning Fine-grained Pruned Sparse Model
    Epoch 0 Sparse Accuracy 0.78% / Best Sparse Accuracy: 0.78%


In [27]:
model.load_state_dict(best_sparse_checkpoint['state_dict'])

t0=time.time()
train_acc, pruned_finetune_model_accuracy=test(model)
t1=time.time()
t_pruned_finetune_model=t1 - t0
###
pruned_finetune_model_size = get_model_size(model)
num_parm_pruned_finetune_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_finetune_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_finetune_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_finetune_model}")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_finetune_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")

90.0% sparse model has accuracy on test set=0.79%
90.0% sparse model has size=0.26 MiB
The time inference of 90.0% sparse model is =18.057109117507935
The number of parametrs of 90.0% sparse model is:6935
90.0% sparse model has size=0.26 MiB, which is 1.00X smaller than the 0.26 MiB dense model


## Manual Measurement

In [38]:
import statistics as stat

sparsity=0.9
Eva_final=dict()


Base_model_accuracy=[]
T_base_model=[]
Num_parm_base_model=[]
Base_model_size=[]

Pruned_model_accuracy=[]
T_pruned_model=[]
Num_parm_pruned_model=[]
Pruned_model_size=[]

Pruned_finetune_model_accuracy=[]
T_pruned_finetune_model=[]
Num_parm_pruned_finetune_model=[]
Pruned_finetune_model_size=[]

In [42]:
Eva=dict()

print(f'Training and evaluation before pruning ')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, data = Net().to(device), data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-3)
for epoch in range(1, 50):
    train(callbacks = None)
    train_acc,  test_acc = test(model)
    if epoch%20==0:
        print(f'Epoch: {epoch:03d}, Train: {train_acc:.4f}, Test: {test_acc:.4f}')

best_checkpoint = dict()
best_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
model.load_state_dict(best_checkpoint['state_dict'])
recover_model = lambda: model.load_state_dict(best_checkpoint['state_dict'])

t0=time.time()
train_acc, base_model_accuracy=test(model)
t1=time.time()
t_base_model=t1 - t0
###
base_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"dense model has accuracy on test set={base_model_accuracy:.2f}%")
print(f"dense model has size={base_model_size/MiB:.2f} MiB")
print(f"The time inference of base model is ={t_base_model}") 
print(f"The number of parametrs of base model is:{num_parm_base_model}")   

#Update my Eva dictionary
Eva.update({'base model accuracy': base_model_accuracy,
            'time inference of base model': t_base_model,
            'number parmameters of base model': num_parm_base_model,
            'size of base model': base_model_size})


print('_______________________________________________________')
print(f'Prune the Model and Re-Evaluate the Accuracy')
recover_model()
pruner = FineGrainedPruner(model, sparsity)
pruner.apply(model)


t0=time.time()
train_acc, pruned_model_accuracy=test(model)
t1=time.time()
t_pruned_model=t1 - t0
###
pruned_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_model}")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")



#Update my Eva dictionary
Eva.update({'pruned model accuracy': pruned_model_accuracy,
            'time inference of pruned model': t_pruned_model,
            'number parmameters of pruned model': num_parm_pruned_model,
            'size of pruned model': pruned_model_size})



print('_______________________________________________________')
print(f'Finetuning Fine-grained Pruned Sparse Model')

best_sparse_checkpoint = dict()
best_sparse_accuracy = 0
num_finetune_epochs=50

for epoch in range(num_finetune_epochs):
    # At the end of each train iteration, we have to apply the pruning mask
    #    to keep the model sparse during the training
    train(callbacks=[lambda: pruner.apply(model)])
    train_acc, accuracy = test(model)

    is_best = accuracy > best_sparse_accuracy
    if is_best:
        best_sparse_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
        best_sparse_accuracy = accuracy

    if epoch % 20 == 0:
         print(f'Epoch {epoch} Sparse Accuracy {accuracy:.2f}% / Best Sparse Accuracy: {best_sparse_accuracy:.2f}%')

model.load_state_dict(best_sparse_checkpoint['state_dict'])


t0=time.time()
train_acc, pruned_finetune_model_accuracy=test(model)
t1=time.time()
t_pruned_finetune_model=t1 - t0
###
pruned_finetune_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_finetune_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_finetune_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_finetune_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_finetune_model}")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_finetune_model_size:.2f}X smaller than "
  f"the {base_model_size/MiB:.2f} MiB dense model")


 #Update my Eva dictionary
Eva.update({'pruned and finetune model accuracy': pruned_finetune_model_accuracy,
            'time inference of pruned and finetune model': t_pruned_finetune_model,
            'number parmameters of pruned and finetune model': num_parm_pruned_finetune_model,
            'size of pruned and finetune model':  pruned_finetune_model_size})



Base_model_accuracy.append(Eva['base model accuracy'])
T_base_model.append(Eva['time inference of base model'])
Num_parm_base_model.append(int(Eva['number parmameters of base model']))
Base_model_size.append(int(Eva['size of base model']))

Pruned_model_accuracy.append(Eva['pruned model accuracy'])
T_pruned_model.append(Eva['time inference of pruned model'])
Num_parm_pruned_model.append(int(Eva['number parmameters of pruned model']))
Pruned_model_size.append(int(Eva['size of pruned model']))

Pruned_finetune_model_accuracy.append(Eva['pruned and finetune model accuracy'])
T_pruned_finetune_model.append(Eva['time inference of pruned and finetune model'])
Num_parm_pruned_finetune_model.append(int(Eva['number parmameters of pruned and finetune model']))
Pruned_finetune_model_size.append(int(Eva['size of pruned and finetune model']))

Training and evaluation before pruning 
Epoch: 020, Train: 0.9479, Test: 0.8800
Epoch: 040, Train: 0.9737, Test: 0.8780
dense model has accuracy on test set=0.89%
dense model has size=0.26 MiB
The time inference of base model is =19.249125242233276
The number of parametrs of base model is:69143
_______________________________________________________
Prune the Model and Re-Evaluate the Accuracy
90.0% sparse model has accuracy on test set=0.74%
90.0% sparse model has size=0.03 MiB
The time inference of 90.0% sparse model is =21.08717370033264
The number of parametrs of 90.0% sparse model is:6935
90.0% sparse model has size=0.03 MiB, which is 9.97X smaller than the 0.26 MiB dense model
_______________________________________________________
Finetuning Fine-grained Pruned Sparse Model
Epoch 0 Sparse Accuracy 0.75% / Best Sparse Accuracy: 0.75%
Epoch 20 Sparse Accuracy 0.85% / Best Sparse Accuracy: 0.85%
Epoch 40 Sparse Accuracy 0.87% / Best Sparse Accuracy: 0.87%
90.0% sparse model has acc

In [43]:

print(f"Base_model_accuracy:{Base_model_accuracy}")
print(f"T_base_model:{T_base_model}")
print(f"Num_parm_base_model:{Num_parm_base_model}")
print(f"Base_model_size:{Base_model_size}")


print(f"Pruned_model_accuracy:{Pruned_model_accuracy}")
print(f"T_pruned_model:{T_pruned_model}")
print(f"Num_parm_Pruned_model:{Num_parm_pruned_model}")
print(f"Pruned_model_size:{Pruned_model_size}")

print(f"Pruned_finetune_model_accuracy:{Pruned_finetune_model_accuracy}")
print(f"T_Pruned_finetune_model:{T_pruned_finetune_model}")
print(f"Num_parm_Pruned_finetune_model:{Num_parm_pruned_finetune_model}")
print(f"Pruned_finetune_model_size:{Pruned_finetune_model_size}")

Base_model_accuracy:[0.882, 0.888]
T_base_model:[19.95966362953186, 19.249125242233276]
Num_parm_base_model:[69143, 69143]
Base_model_size:[2212576, 2212576]
Pruned_model_accuracy:[0.764, 0.742]
T_pruned_model:[23.04162287712097, 21.08717370033264]
Num_parm_Pruned_model:[6935, 6935]
Pruned_model_size:[221920, 221920]
Pruned_finetune_model_accuracy:[0.868, 0.876]
T_Pruned_finetune_model:[16.74787211418152, 18.19109272956848]
Num_parm_Pruned_finetune_model:[6935, 6935]
Pruned_finetune_model_size:[221920, 221920]


In [66]:
Eva_final=dict()
base_model_accuracy_mean = stat.mean(Base_model_accuracy)
base_model_accuracy_std =  stat.stdev(Base_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)

Eva_final.update({'base model accuracy':float(format(base_model_accuracy_mean, '.4f'))})
                 
t_base_model_mean =stat.mean(T_base_model)
#t_base_model_std =t_base_model.std()
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of base model':float(format(t_base_model_mean, '.6f'))})

num_parm_base_model_mean = stat.mean(Num_parm_base_model)
#num_parm_base_model_std = num_parm_base_model.std()
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of base model':num_parm_base_model_mean})

base_model_size_mean = stat.mean(Base_model_size)
#base_model_size_std = base_model_size.std()
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'base_model_size':base_model_size_mean})

#################################

pruned_model_accuracy_mean =stat.mean(Pruned_model_accuracy)
pruned_model_accuracy_std = stat.stdev(Pruned_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned model accuracy':float(format(pruned_model_accuracy_mean, '.4f'))})
                 

t_pruned_model_mean = stat.mean(T_pruned_model)
#t_base_model_std =t_dence_model.std()
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of pruned model':float(format(t_pruned_model_mean, '.6f'))})

num_parm_pruned_model_mean = stat.mean(Num_parm_pruned_model)
#num_parm_base_model_std = num_parm_base_model.std()
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of pruned model':num_parm_pruned_model_mean})

pruned_model_size_mean =stat.mean( Pruned_model_size)
#base_model_size_std = base_model_size.std()
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned model size':pruned_model_size_mean})

#################################
pruned_finetune_model_accuracy_mean =stat.mean(Pruned_finetune_model_accuracy)
pruned_finetune_model_accuracy_std = stat.stdev(Pruned_finetune_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned finetune model accuracy':float(format(pruned_finetune_model_accuracy_mean, '.4f'))})
                 

t_pruned_finetune_model_mean =stat.mean(T_pruned_finetune_model)
#t_base_model_std =t_dence_model.std()
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of pruned finetune model':float(format(t_pruned_finetune_model_mean,'.6f'))})

num_parm_pruned_finetune_model_mean =stat.mean(Num_parm_pruned_finetune_model)
#num_parm_base_model_std = num_parm_base_model.std()
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of pruned finetune model':num_parm_pruned_finetune_model_mean})

pruned_finetune_model_size_mean = stat.mean(Pruned_finetune_model_size)
#base_model_size_std = base_model_size.std()
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned finetune model size':pruned_finetune_model_size_mean})


#################################


print(f"All measurement about pruning process of sparsity:{sparsity*100}% ")   
Eva_final

All measurement about pruning process of sparsity:90.0% 


{'base model accuracy': 0.9514,
 'time inference of base model': 0.012501,
 'number parmameters of base model': 23063,
 'base_model_size': 738016,
 'pruned model accuracy': 0.377,
 'time inference of pruned model': 0.013153,
 'number parmameters of pruned model': 2327,
 'pruned model size': 74464,
 'pruned finetune model accuracy': 0.8304,
 'time inference of pruned finetune model': 0.015108,
 'number parmameters of pruned finetune model': 2327,
 'pruned finetune model size': 74464}

In [67]:
Cora_Node_90={'base model accuracy': 0.9514,
 'time inference of base model': 0.012501,
 'number parmameters of base model': 23063,
 'base_model_size': 738016,
 'pruned model accuracy': 0.377,
 'time inference of pruned model': 0.013153,
 'number parmameters of pruned model': 2327,
 'pruned model size': 74464,
 'pruned finetune model accuracy': 0.8304,
 'time inference of pruned finetune model': 0.015108,
 'number parmameters of pruned finetune model': 2327,
 'pruned finetune model size': 74464}